# Kaggle Test Set Prediction

**Objective:** Load the labelled training data prepared in Notebook 2, apply the same classification preprocessing to Kaggle's unlabelled test data, train the selected TF-IDF model on all labelled rows, and create `submission.csv`.

The Kaggle test set is not part of `train_dataset_preprocessed.pkl`; this notebook loads it separately from `Datasets/test.csv`. Test IDs are preserved and aligned to `sample_submission.csv` when writing the output.

## 1. Load the prepared training data and Kaggle test data

Notebook 2 saved the processed, labelled training rows. The Kaggle test CSV contains the test lyrics and IDs, but no genre labels.

In [24]:
import pickle

with open('test_for_classification.pkl', 'rb') as f:
    test_for_classification = pickle.load(f)

print("Test dataset loaded successfully!")

Test dataset loaded successfully!


In [25]:

with open('train_for_classification.pkl', 'rb') as f:
    train_for_classification = pickle.load(f)

print("Train dataset loaded successfully!")

Train dataset loaded successfully!


In [26]:
train_for_classification.shape

(134528, 10)

In [27]:
import pickle
import re

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import SGDClassifier



assert "lyrics_processed_for_classification" in train_dataset.columns
assert "tag" in train_dataset.columns
assert "lyrics" in test_dataset.columns

print(f"Labelled training rows: {len(train_for_classification):,}")
print(f"Kaggle test rows: {len(test_for_classification):,}")

Labelled training rows: 134,528
Kaggle test rows: 33,742


## 2. Apply the classification preprocessing to Kaggle test lyrics

Use the same regex rules as Notebook 2, then the shared tokenization, stopword-removal, and lemmatization pipeline. Do not remove any test rows: Kaggle expects one prediction for every test ID.

In [28]:
test_for_classification

,title,artist,year,views,features,raw_lyrics,lyrics_cleaned,lyrics_normalized
id,,,,,,,,
3996359,Like You Do,Jimmie Allen,2018,825,{},[Verse 1]\r\nYour head's on my chest\r\nThis b...,Your heads on my chest This beds all a mess Em...,head chest bed mess empty bottle red wine tell...
3553166,This Building is Falling Down,The Taxpayers,2009,400,{},Epilogue: A sound off from simple people with ...,Epilogue A sound off from simple people with s...,epilogue sound simple people simple theory fuc...
2998742,My Private Joy,Dion and The Belmonts,1960,229,{},[Verse 1]\r\nOh well Susie is a doozy with a p...,Oh well Susie is a doozy with a pony tail She ...,oh well susie doozy pony tail wear gold polish...
3993470,Vegan,Yung Kurry,2018,1041,{},Hook\r\nShe a vegan but she eat my meat with n...,Hook She a vegan but she eat my meat with no s...,vegan eat meat season take look wrist ooh shit...
5488256,To The Boy I Used To Know,Angelina Cruz,2018,63,{},To the boy I used to know\r\nI wonder how do y...,To the boy I used to know I wonder how do you ...,boy use wonder boy use still think old break m...
...,...,...,...,...,...,...,...,...
1149534,My Life With You,Celtic Thunder,2015,32,{},When I was a younger man\r\nWhen I was a silly...,When I was a younger man When I was a silly bo...,wa young man wa silly boy need thing wa strong...
2473358,Work it out,The Jacka,2014,488,{Husalah},[Intro]\r\nNigga did you take my money?\r\nHeh...,Nigga did you take my money Hehehe thats right...,nigga take money hehehe thats right lil bitch ...
5254514,No More,Darling West,2016,20,{},The moon is hanging high above\r\nA long and d...,The moon is hanging high above A long and dust...,moon hang high long dusty red dirt road blood ...


## 3. Fit TF-IDF and the selected classifier on all labelled data

Validation selected the combined word + character TF-IDF features with SGD Logistic (`alpha=0.0001`, `class_weight="balanced"`). Fit both TF-IDF vectorizers and the classifier on all labelled rows, then transform and predict the Kaggle test rows. The test data is never used for fitting.

In [29]:
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, f1_score

# 1. Preparar os dados de treino e teste
X_train_full = train_for_classification["lyrics_normalized"].fillna("").astype(str)
y_train_full = train_for_classification["tag"]
X_test = test_for_classification["lyrics_normalized"].fillna("").astype(str)

# 2. Configurar o TfidfVectorizer apenas com palavras (unigrams + bigrams)
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b\w+\b",
    min_df=3,
    max_features=30000
)

print("A vetorizar os dados de treino...")
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_full)

print("A vetorizar os dados de teste...")
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# 3. Configurar o SGDClassifier com pesos balanceados
final_model = SGDClassifier(
    loss="log_loss",
    class_weight="balanced",
    max_iter=2000,
    random_state=42,
    alpha=0.0001,
)

print("A treinar o modelo SGDClassifier...")
final_model.fit(X_train_tfidf, y_train_full)

# 4. Previsões e métricas de avaliação no treino (cross-val ou holdout se tiveres)
test_predictions = final_model.predict(X_test_tfidf)

print("Previsões criadas com sucesso:", len(test_predictions))

A vetorizar os dados de treino...
A vetorizar os dados de teste...
A treinar o modelo SGDClassifier...
Previsões criadas com sucesso: 33742


## 4. Create and validate the Kaggle submission

Use the IDs and column order from `sample_submission.csv` so the output matches Kaggle's expected format.

In [30]:
sample_submission = pd.read_csv("Datasets/sample_submission.csv")

predictions_by_id = pd.Series(test_predictions, index=test_for_classification.index, name="tag")
submission = (
    predictions_by_id.reindex(sample_submission["id"])
    .rename_axis("id")
    .reset_index()
)

assert len(submission) == len(sample_submission)
assert submission["tag"].notna().all()
assert submission.columns.tolist() == ["id", "tag"]
assert submission["id"].equals(sample_submission["id"])

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
display(submission.head())

Saved submission.csv


,id,tag
0,3230388,misc
1,3658643,pop
2,1270714,pop
3,3354312,rap
4,493974,pop
